# osm2lod – Linked Data Exploration

This notebook demonstrates the value of Linked Data by querying six osm2lod TTL exports with SPARQL and visualising the results.

**Datasets:**
- 🪨 **Ogham Stones** – Early medieval inscribed stones (Ireland / Northern Ireland)
- 💧 **Holy Wells** – Sacred water sites (Ireland / Northern Ireland)
- 🦴 **CI Findspots** – Palaeolithic cave sites
- 🌋 **Drillcores / Maars** – Volcanic drill-core profiles
- 📐 **Benchmarks** – Ordnance survey points
- 🌀 **SISAL Sites** – Speleothem cave sites (global)
- 🏛️ **Roman Sites** – Roman archaeological sites

**Why Linked Data?** Each entity carries stable URIs, geographic coordinates, and typed links to Wikidata — enabling cross-dataset queries, spatial analysis, and federation with external knowledge graphs.

## 0 — Setup

In [ ]:
# Standard library
import re
from pathlib import Path

# Data
import pandas as pd

# RDF / SPARQL
from rdflib import ConjunctiveGraph, Graph, Namespace
from rdflib.namespace import RDF, RDFS, OWL

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import folium
from folium.plugins import MarkerCluster

# Jupyter
from IPython.display import display, HTML

# ── Namespaces ──────────────────────────────────────────────────────────────
OSM2LOD    = Namespace("http://research-squirrel-engineers.github.io/osm2lod/")
GEOSPARQL  = Namespace("http://www.opengis.net/ont/geosparql#")
DCTERMS    = Namespace("http://purl.org/dc/terms/")
WD         = Namespace("http://wikidata.org/entity/")
FOAF       = Namespace("http://xmlns.com/foaf/0.1/")

# ── File paths ───────────────────────────────────────────────────────────────
# Adjust if your TTL files are in a different location
DATA_DIR = Path(".")   # same folder as this notebook

TTL_FILES = {
    "ogham":      next(DATA_DIR.glob("osm_export_ogham_*.ttl")),
    "holywells":  next(DATA_DIR.glob("osm_export_holywells_*.ttl")),
    "ci":         next(DATA_DIR.glob("osm_export_ci_*.ttl")),
    "benchmarks": next(DATA_DIR.glob("osm_export_benchmarks_*.ttl")),
    "sisal":      next(DATA_DIR.glob("osm_export_sisal_*.ttl")),
    "romansites": next(DATA_DIR.glob("osm_export_romansites_*.ttl")),
}

print("Files found:")
for k, v in TTL_FILES.items():
    print(f"  {k:12s} → {v.name}")

In [ ]:
# ── Load all graphs ──────────────────────────────────────────────────────────
GRAPHS: dict[str, Graph] = {}
for name, path in TTL_FILES.items():
    g = Graph()
    g.parse(path, format="turtle")
    GRAPHS[name] = g
    print(f"  {name:12s}: {len(g):5d} triples")

# Combined graph for cross-dataset queries
ALL = ConjunctiveGraph()
for g in GRAPHS.values():
    for triple in g:
        ALL.add(triple)
print(f"\n  {'TOTAL':12s}: {len(ALL):5d} triples across all datasets")

---
## 1 — Overview: Items and Linked Data Coverage

**SPARQL query:** For each dataset, count total items and items with a Wikidata link (`owl:sameAs`).

In [ ]:
SPARQL_OVERVIEW = """
PREFIX osm2lod:   <http://research-squirrel-engineers.github.io/osm2lod/>
PREFIX owl:       <http://www.w3.org/2002/07/owl#>

SELECT (COUNT(?item) AS ?total) (COUNT(?wd) AS ?with_wikidata)
WHERE {
  ?item osm2lod:exportType ?et .
  OPTIONAL { ?item owl:sameAs ?wd .
             FILTER(STRSTARTS(STR(?wd), "http://wikidata.org/")) }
}
"""

rows = []
for name, g in GRAPHS.items():
    res = list(g.query(SPARQL_OVERVIEW))
    total = int(res[0][0])
    with_wd = int(res[0][1])
    rows.append({"Dataset": name, "Total items": total,
                 "With Wikidata": with_wd,
                 "Coverage %": round(100 * with_wd / total, 1) if total else 0})

df_overview = pd.DataFrame(rows).sort_values("Total items", ascending=False)
display(df_overview.to_html(index=False))

In [ ]:
# ── Chart: Total items + Wikidata coverage ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("osm2lod Datasets – Overview", fontsize=14, fontweight="bold")

COLORS = {
    "ogham":      "#c0392b",
    "holywells":  "#2980b9",
    "ci":         "#8e44ad",
    "benchmarks": "#27ae60",
    "sisal":      "#e67e22",
    "romansites": "#7f8c8d",
}

names  = df_overview["Dataset"].tolist()
totals = df_overview["Total items"].tolist()
wds    = df_overview["With Wikidata"].tolist()
colors = [COLORS[n] for n in names]

# Left: item counts
ax0 = axes[0]
bars = ax0.barh(names, totals, color=colors, alpha=0.85)
ax0.bar_label(bars, padding=4, fontsize=10)
ax0.set_xlabel("Number of items")
ax0.set_title("Total items per dataset")
ax0.invert_yaxis()
ax0.spines[["top","right"]].set_visible(False)

# Right: Wikidata coverage %
ax1 = axes[1]
pcts = df_overview["Coverage %"].tolist()
bars2 = ax1.barh(names, pcts, color=colors, alpha=0.85)
ax1.bar_label(bars2, fmt="%.1f%%", padding=4, fontsize=10)
ax1.set_xlabel("% items linked to Wikidata")
ax1.set_title("Wikidata linkage coverage")
ax1.set_xlim(0, 110)
ax1.invert_yaxis()
ax1.spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.show()

---
## 2 — Holy Wells (Ireland / Northern Ireland)

**SPARQL queries:**
1. All wells with coordinates and label → map
2. Denomination breakdown → pie chart

In [ ]:
SPARQL_HOLYWELLS_MAP = """
PREFIX osm2lod:  <http://research-squirrel-engineers.github.io/osm2lod/>
PREFIX geosparql:<http://www.opengis.net/ont/geosparql#>
PREFIX rdfs:     <http://www.w3.org/2000/01/rdf-schema#>
PREFIX foaf:     <http://xmlns.com/foaf/0.1/>
PREFIX owl:      <http://www.w3.org/2002/07/owl#>

SELECT ?item ?label ?wkt ?osmurl ?denomination ?wd
WHERE {
  ?item osm2lod:exportType "holywells" ;
        geosparql:hasGeometry ?geom .
  ?geom geosparql:asWKT ?wkt .
  OPTIONAL { ?item rdfs:label ?label . FILTER(lang(?label) = "en") }
  OPTIONAL { ?item foaf:primaryTopic ?osmurl }
  OPTIONAL { ?item osm2lod:osmtag__denomination ?denomination }
  OPTIONAL { ?item owl:sameAs ?wd .
             FILTER(STRSTARTS(STR(?wd), "http://wikidata.org/")) }
}
"""

def parse_wkt(wkt_str):
    """Extract (lat, lon) from '<CRS> POINT(lon lat)' string."""
    m = re.search(r"POINT\(([\d.\-]+)\s+([\d.\-]+)\)", str(wkt_str))
    if m:
        lon, lat = float(m.group(1)), float(m.group(2))
        return lat, lon
    return None, None

hw_rows = []
for r in GRAPHS["holywells"].query(SPARQL_HOLYWELLS_MAP):
    lat, lon = parse_wkt(r.wkt)
    hw_rows.append({
        "label":       str(r.label) if r.label else "(unnamed)",
        "lat":         lat,
        "lon":         lon,
        "osm_url":     str(r.osmurl) if r.osmurl else "",
        "denomination":str(r.denomination) if r.denomination else "unknown",
        "wikidata":    str(r.wd) if r.wd else "",
    })

df_hw = pd.DataFrame(hw_rows).dropna(subset=["lat","lon"])
print(f"Holy Wells: {len(df_hw)} items with coordinates")

In [ ]:
# ── Map: Holy Wells ──────────────────────────────────────────────────────────
m_hw = folium.Map(location=[53.5, -8.0], zoom_start=7,
                  tiles="OpenStreetMap")
cluster = MarkerCluster(name="Holy Wells").add_to(m_hw)

for _, row in df_hw.iterrows():
    has_wd = bool(row["wikidata"])
    color  = "blue" if has_wd else "lightgray"
    popup_parts = [f"<b>{row['label']}</b>"]
    if row["denomination"] != "unknown":
        popup_parts.append(f"Denomination: {row['denomination']}")
    if row["osm_url"]:
        popup_parts.append(f'<a href="{row["osm_url"]}" target="_blank">OSM ↗</a>')
    if has_wd:
        wd_url = row["wikidata"].replace("http://wikidata.org/entity/",
                                         "https://www.wikidata.org/wiki/")
        popup_parts.append(f'<a href="{wd_url}" target="_blank">Wikidata ↗</a>')

    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=5, color=color, fill=True, fill_opacity=0.7,
        popup=folium.Popup("<br>".join(popup_parts), max_width=220),
        tooltip=row["label"]
    ).add_to(cluster)

# Legend
legend_html = """
<div style="position:fixed;bottom:30px;left:30px;z-index:1000;
     background:white;padding:10px 15px;border-radius:6px;
     border:1px solid #ccc;font-size:13px;">
  <b>Holy Wells</b><br>
  <span style="color:steelblue;">●</span> Linked to Wikidata<br>
  <span style="color:lightgray;">●</span> OSM only
</div>"""
m_hw.get_root().html.add_child(folium.Element(legend_html))

display(HTML(f"<p><b>Holy Wells map</b> — {len(df_hw)} sites "
             f"({df_hw['wikidata'].astype(bool).sum()} linked to Wikidata)</p>"))
m_hw

In [ ]:
# ── Chart: Denomination breakdown ────────────────────────────────────────────
denom_counts = df_hw["denomination"].value_counts()

fig, ax = plt.subplots(figsize=(7, 5))
wedge_colors = ["#2980b9", "#85c1e9", "#abebc6", "#f9e79f", "#d5dbdb"]
wedges, texts, autotexts = ax.pie(
    denom_counts.values,
    labels=denom_counts.index,
    autopct="%1.1f%%",
    colors=wedge_colors[:len(denom_counts)],
    startangle=140,
    pctdistance=0.75
)
for t in autotexts:
    t.set_fontsize(10)
ax.set_title("Holy Wells – Denomination (OSM tag)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 3 — Ogham Stones

**SPARQL:** All Ogham stones with label, Wikidata link, and coordinates.

In [ ]:
SPARQL_OGHAM = """
PREFIX osm2lod:  <http://research-squirrel-engineers.github.io/osm2lod/>
PREFIX geosparql:<http://www.opengis.net/ont/geosparql#>
PREFIX rdfs:     <http://www.w3.org/2000/01/rdf-schema#>
PREFIX foaf:     <http://xmlns.com/foaf/0.1/>
PREFIX owl:      <http://www.w3.org/2002/07/owl#>

SELECT ?item ?label ?wkt ?osmurl ?wd
WHERE {
  ?item a osm2lod:OghamStone ;
        geosparql:hasGeometry ?geom .
  ?geom geosparql:asWKT ?wkt .
  OPTIONAL { ?item rdfs:label ?label . FILTER(lang(?label) = "en") }
  OPTIONAL { ?item foaf:primaryTopic ?osmurl }
  OPTIONAL { ?item owl:sameAs ?wd .
             FILTER(STRSTARTS(STR(?wd), "http://wikidata.org/")) }
}
ORDER BY ?label
"""

og_rows = []
for r in GRAPHS["ogham"].query(SPARQL_OGHAM):
    lat, lon = parse_wkt(r.wkt)
    og_rows.append({
        "label":    str(r.label) if r.label else "(unnamed)",
        "lat":      lat, "lon": lon,
        "osm_url":  str(r.osmurl) if r.osmurl else "",
        "wikidata": str(r.wd) if r.wd else "",
    })

df_og = pd.DataFrame(og_rows).dropna(subset=["lat","lon"])
print(f"Ogham Stones: {len(df_og)} items | "
      f"{df_og['wikidata'].astype(bool).sum()} linked to Wikidata")

In [ ]:
# ── Map: Ogham Stones ────────────────────────────────────────────────────────
m_og = folium.Map(location=[52.5, -8.2], zoom_start=7, tiles="OpenStreetMap")

for _, row in df_og.iterrows():
    has_wd = bool(row["wikidata"])
    color  = "#c0392b" if has_wd else "#e8a49a"
    popup_parts = [f"<b>{row['label']}</b>"]
    if row["osm_url"]:
        popup_parts.append(f'<a href="{row["osm_url"]}" target="_blank">OSM ↗</a>')
    if has_wd:
        wd_url = row["wikidata"].replace("http://wikidata.org/entity/",
                                         "https://www.wikidata.org/wiki/")
        popup_parts.append(f'<a href="{wd_url}" target="_blank">Wikidata ↗</a>')

    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=6, color=color, fill=True, fill_opacity=0.8,
        popup=folium.Popup("<br>".join(popup_parts), max_width=220),
        tooltip=row["label"]
    ).add_to(m_og)

legend_html = """
<div style="position:fixed;bottom:30px;left:30px;z-index:1000;
     background:white;padding:10px 15px;border-radius:6px;
     border:1px solid #ccc;font-size:13px;">
  <b>Ogham Stones</b><br>
  <span style="color:#c0392b;">●</span> Linked to Wikidata<br>
  <span style="color:#e8a49a;">●</span> OSM only
</div>"""
m_og.get_root().html.add_child(folium.Element(legend_html))
m_og

---
## 4 — SISAL Cave Sites (Global)

**SPARQL:** All SISAL sites with coordinates, label, and optional Wikidata link — visualised on a world map.

In [ ]:
SPARQL_SISAL = """
PREFIX osm2lod:  <http://research-squirrel-engineers.github.io/osm2lod/>
PREFIX geosparql:<http://www.opengis.net/ont/geosparql#>
PREFIX rdfs:     <http://www.w3.org/2000/01/rdf-schema#>
PREFIX foaf:     <http://xmlns.com/foaf/0.1/>
PREFIX owl:      <http://www.w3.org/2002/07/owl#>

SELECT ?item ?label ?wkt ?osmurl ?wd ?natural
WHERE {
  ?item a osm2lod:SisalSite ;
        geosparql:hasGeometry ?geom .
  ?geom geosparql:asWKT ?wkt .
  OPTIONAL { ?item rdfs:label ?label . FILTER(lang(?label) = "en") }
  OPTIONAL { ?item foaf:primaryTopic ?osmurl }
  OPTIONAL { ?item owl:sameAs ?wd .
             FILTER(STRSTARTS(STR(?wd), "http://wikidata.org/")) }
  OPTIONAL { ?item osm2lod:osmtag__natural ?natural }
}
ORDER BY ?label
"""

sisal_rows = []
for r in GRAPHS["sisal"].query(SPARQL_SISAL):
    lat, lon = parse_wkt(r.wkt)
    sisal_rows.append({
        "label":    str(r.label) if r.label else "(unnamed)",
        "lat": lat, "lon": lon,
        "osm_url":  str(r.osmurl) if r.osmurl else "",
        "wikidata": str(r.wd) if r.wd else "",
        "natural":  str(r.natural) if r.natural else "",
    })

df_sisal = pd.DataFrame(sisal_rows).dropna(subset=["lat","lon"])
print(f"SISAL Sites: {len(df_sisal)} items | "
      f"{df_sisal['wikidata'].astype(bool).sum()} linked to Wikidata")

In [ ]:
# ── World Map: SISAL Sites ───────────────────────────────────────────────────
m_sisal = folium.Map(location=[20, 10], zoom_start=2, tiles="OpenStreetMap")
cluster_sisal = MarkerCluster(name="SISAL Sites").add_to(m_sisal)

for _, row in df_sisal.iterrows():
    has_wd = bool(row["wikidata"])
    color  = "#e67e22" if has_wd else "#f0c27f"
    popup_parts = [f"<b>{row['label']}</b>"]
    if row["osm_url"]:
        popup_parts.append(f'<a href="{row["osm_url"]}" target="_blank">OSM ↗</a>')
    if has_wd:
        wd_url = row["wikidata"].replace("http://wikidata.org/entity/",
                                         "https://www.wikidata.org/wiki/")
        popup_parts.append(f'<a href="{wd_url}" target="_blank">Wikidata ↗</a>')

    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=6, color=color, fill=True, fill_opacity=0.85,
        popup=folium.Popup("<br>".join(popup_parts), max_width=220),
        tooltip=row["label"]
    ).add_to(cluster_sisal)

legend_html = """
<div style="position:fixed;bottom:30px;left:30px;z-index:1000;
     background:white;padding:10px 15px;border-radius:6px;
     border:1px solid #ccc;font-size:13px;">
  <b>SISAL Cave Sites</b><br>
  <span style="color:#e67e22;">●</span> Linked to Wikidata<br>
  <span style="color:#f0c27f;">●</span> OSM only
</div>"""
m_sisal.get_root().html.add_child(folium.Element(legend_html))
m_sisal

---
## 5 — Roman Sites

**SPARQL:** All Roman sites with label, archaeological site type, and Wikidata link.

In [ ]:
SPARQL_ROMAN = """
PREFIX osm2lod:  <http://research-squirrel-engineers.github.io/osm2lod/>
PREFIX geosparql:<http://www.opengis.net/ont/geosparql#>
PREFIX rdfs:     <http://www.w3.org/2000/01/rdf-schema#>
PREFIX foaf:     <http://xmlns.com/foaf/0.1/>
PREFIX owl:      <http://www.w3.org/2002/07/owl#>

SELECT ?item ?label ?wkt ?osmurl ?wd ?site_type ?civilization
WHERE {
  ?item a osm2lod:RomanSite ;
        geosparql:hasGeometry ?geom .
  ?geom geosparql:asWKT ?wkt .
  OPTIONAL { ?item rdfs:label ?label . FILTER(lang(?label) = "en") }
  OPTIONAL { ?item foaf:primaryTopic ?osmurl }
  OPTIONAL { ?item owl:sameAs ?wd .
             FILTER(STRSTARTS(STR(?wd), "http://wikidata.org/")) }
  OPTIONAL { ?item osm2lod:osmtag__archaeological_site ?site_type }
  OPTIONAL { ?item osm2lod:osmtag__historic__civilization ?civilization }
}
ORDER BY ?label
"""

roman_rows = []
for r in GRAPHS["romansites"].query(SPARQL_ROMAN):
    lat, lon = parse_wkt(r.wkt)
    roman_rows.append({
        "label":       str(r.label) if r.label else "(unnamed)",
        "lat": lat,    "lon": lon,
        "osm_url":     str(r.osmurl) if r.osmurl else "",
        "wikidata":    str(r.wd) if r.wd else "",
        "site_type":   str(r.site_type) if r.site_type else "unknown",
        "civilization":str(r.civilization) if r.civilization else "untagged",
    })

df_roman = pd.DataFrame(roman_rows).dropna(subset=["lat","lon"])
print(f"Roman Sites: {len(df_roman)} items | "
      f"{df_roman['wikidata'].astype(bool).sum()} linked to Wikidata")
display(df_roman[["label","site_type","civilization","wikidata"]].to_html(index=False))

In [ ]:
# ── Map: Roman Sites ─────────────────────────────────────────────────────────
m_roman = folium.Map(location=[47.5, 8.5], zoom_start=4, tiles="OpenStreetMap")

for _, row in df_roman.iterrows():
    has_wd = bool(row["wikidata"])
    color  = "#7f8c8d" if has_wd else "#bdc3c7"
    popup_parts = [f"<b>{row['label']}</b>",
                   f"Type: {row['site_type']}",
                   f"Civilization: {row['civilization']}"]
    if row["osm_url"]:
        popup_parts.append(f'<a href="{row["osm_url"]}" target="_blank">OSM ↗</a>')
    if has_wd:
        wd_url = row["wikidata"].replace("http://wikidata.org/entity/",
                                         "https://www.wikidata.org/wiki/")
        popup_parts.append(f'<a href="{wd_url}" target="_blank">Wikidata ↗</a>')

    folium.Marker(
        location=[row["lat"], row["lon"]],
        popup=folium.Popup("<br>".join(popup_parts), max_width=240),
        tooltip=row["label"],
        icon=folium.Icon(color="gray", icon="landmark", prefix="fa")
    ).add_to(m_roman)

m_roman

---
## 6 — CI Findspots

**SPARQL:** All CI sites with civilization tag, Wikidata / Wikipedia links.

In [ ]:
SPARQL_CI = """
PREFIX osm2lod:  <http://research-squirrel-engineers.github.io/osm2lod/>
PREFIX geosparql:<http://www.opengis.net/ont/geosparql#>
PREFIX rdfs:     <http://www.w3.org/2000/01/rdf-schema#>
PREFIX foaf:     <http://xmlns.com/foaf/0.1/>
PREFIX owl:      <http://www.w3.org/2002/07/owl#>

SELECT ?item ?label ?wkt ?osmurl ?wd ?civilization ?natural
WHERE {
  ?item a osm2lod:CI_Findspot ;
        geosparql:hasGeometry ?geom .
  ?geom geosparql:asWKT ?wkt .
  OPTIONAL { ?item rdfs:label ?label . FILTER(lang(?label) = "en") }
  OPTIONAL { ?item foaf:primaryTopic ?osmurl }
  OPTIONAL { ?item owl:sameAs ?wd .
             FILTER(STRSTARTS(STR(?wd), "http://wikidata.org/")) }
  OPTIONAL { ?item osm2lod:osmtag__historic__civilization ?civilization }
  OPTIONAL { ?item osm2lod:osmtag__natural ?natural }
}
ORDER BY ?label
"""

ci_rows = []
for r in GRAPHS["ci"].query(SPARQL_CI):
    lat, lon = parse_wkt(r.wkt)
    ci_rows.append({
        "label":       str(r.label) if r.label else "(unnamed)",
        "lat": lat,    "lon": lon,
        "osm_url":     str(r.osmurl) if r.osmurl else "",
        "wikidata":    str(r.wd) if r.wd else "",
        "civilization":str(r.civilization) if r.civilization else "untagged",
        "natural":     str(r.natural) if r.natural else "",
    })

df_ci = pd.DataFrame(ci_rows).dropna(subset=["lat","lon"])
print(f"CI Findspots: {len(df_ci)} items")
display(df_ci[["label","civilization","natural","wikidata"]].to_html(index=False))

In [ ]:
# ── Map: CI Findspots ────────────────────────────────────────────────────────
m_ci = folium.Map(location=[42, 20], zoom_start=4, tiles="OpenStreetMap")

for _, row in df_ci.iterrows():
    has_wd = bool(row["wikidata"])
    color  = "#8e44ad" if has_wd else "#c39bd3"
    popup_parts = [f"<b>{row['label']}</b>"]
    if row["civilization"] != "untagged":
        popup_parts.append(f"Civilization: {row['civilization']}")
    if row["natural"]:
        popup_parts.append(f"Natural: {row['natural']}")
    if row["osm_url"]:
        popup_parts.append(f'<a href="{row["osm_url"]}" target="_blank">OSM ↗</a>')
    if has_wd:
        wd_url = row["wikidata"].replace("http://wikidata.org/entity/",
                                         "https://www.wikidata.org/wiki/")
        popup_parts.append(f'<a href="{wd_url}" target="_blank">Wikidata ↗</a>')

    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=8, color=color, fill=True, fill_opacity=0.85,
        popup=folium.Popup("<br>".join(popup_parts), max_width=240),
        tooltip=row["label"]
    ).add_to(m_ci)

m_ci

---
## 7 — Benchmarks

**SPARQL:** Survey points with Wikimedia Commons photo links and structure type.

In [ ]:
SPARQL_BENCHMARKS = """
PREFIX osm2lod:  <http://research-squirrel-engineers.github.io/osm2lod/>
PREFIX geosparql:<http://www.opengis.net/ont/geosparql#>
PREFIX rdfs:     <http://www.w3.org/2000/01/rdf-schema#>
PREFIX foaf:     <http://xmlns.com/foaf/0.1/>

SELECT ?item ?label ?wkt ?osmurl ?commons ?structure ?survey_date
WHERE {
  ?item a osm2lod:Benchmark ;
        geosparql:hasGeometry ?geom .
  ?geom geosparql:asWKT ?wkt .
  OPTIONAL { ?item rdfs:label ?label . FILTER(lang(?label) = "en") }
  OPTIONAL { ?item foaf:primaryTopic ?osmurl }
  OPTIONAL { ?item osm2lod:osmtag__wikimedia_commons ?commons }
  OPTIONAL { ?item osm2lod:osmtag__survey_point__structure ?structure }
  OPTIONAL { ?item osm2lod:osmtag__survey__date ?survey_date }
}
ORDER BY ?label
"""

bm_rows = []
for r in GRAPHS["benchmarks"].query(SPARQL_BENCHMARKS):
    lat, lon = parse_wkt(r.wkt)
    bm_rows.append({
        "label":       str(r.label) if r.label else "(unnamed)",
        "lat": lat,    "lon": lon,
        "osm_url":     str(r.osmurl) if r.osmurl else "",
        "commons":     str(r.commons) if r.commons else "",
        "structure":   str(r.structure) if r.structure else "unknown",
        "survey_date": str(r.survey_date) if r.survey_date else "",
    })

df_bm = pd.DataFrame(bm_rows).dropna(subset=["lat","lon"])
print(f"Benchmarks: {len(df_bm)} items | "
      f"{df_bm['commons'].astype(bool).sum()} with Wikimedia Commons photo")

In [ ]:
# ── Map: Benchmarks with Commons photos in popups ────────────────────────────
m_bm = folium.Map(location=[53.0, -7.5], zoom_start=7, tiles="OpenStreetMap")

for _, row in df_bm.iterrows():
    has_photo = bool(row["commons"])
    color = "#27ae60" if has_photo else "#a9dfbf"
    popup_parts = [f"<b>{row['label']}</b>",
                   f"Structure: {row['structure']}"]
    if row["survey_date"]:
        popup_parts.append(f"Surveyed: {row['survey_date']}")
    if row["osm_url"]:
        popup_parts.append(f'<a href="{row["osm_url"]}" target="_blank">OSM ↗</a>')
    if has_photo:
        popup_parts.append(f'<a href="{row["commons"]}" target="_blank">📷 Commons ↗</a>')

    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=7, color=color, fill=True, fill_opacity=0.85,
        popup=folium.Popup("<br>".join(popup_parts), max_width=260),
        tooltip=row["label"]
    ).add_to(m_bm)

legend_html = """
<div style="position:fixed;bottom:30px;left:30px;z-index:1000;
     background:white;padding:10px 15px;border-radius:6px;
     border:1px solid #ccc;font-size:13px;">
  <b>Benchmarks</b><br>
  <span style="color:#27ae60;">●</span> With Commons photo<br>
  <span style="color:#a9dfbf;">●</span> No photo yet
</div>"""
m_bm.get_root().html.add_child(folium.Element(legend_html))
m_bm

---
## 8 — Cross-Dataset: All Items with Wikidata Links

**SPARQL on the combined graph:** Retrieve every entity that has an `owl:sameAs` link to Wikidata, across all six datasets — on one shared map.

This is the core Linked Data argument: all datasets share the same RDF vocabulary, so they can be queried together without any ETL.

In [ ]:
SPARQL_ALL_WD = """
PREFIX osm2lod:  <http://research-squirrel-engineers.github.io/osm2lod/>
PREFIX geosparql:<http://www.opengis.net/ont/geosparql#>
PREFIX rdfs:     <http://www.w3.org/2000/01/rdf-schema#>
PREFIX foaf:     <http://xmlns.com/foaf/0.1/>
PREFIX owl:      <http://www.w3.org/2002/07/owl#>

SELECT ?item ?label ?wkt ?exportType ?wd
WHERE {
  ?item osm2lod:exportType ?exportType ;
        geosparql:hasGeometry ?geom ;
        owl:sameAs ?wd .
  ?geom geosparql:asWKT ?wkt .
  FILTER(STRSTARTS(STR(?wd), "http://wikidata.org/"))
  OPTIONAL { ?item rdfs:label ?label . FILTER(lang(?label) = "en") }
}
ORDER BY ?exportType ?label
"""

all_rows = []
for r in ALL.query(SPARQL_ALL_WD):
    lat, lon = parse_wkt(r.wkt)
    all_rows.append({
        "label":      str(r.label) if r.label else "(unnamed)",
        "lat": lat,   "lon": lon,
        "exportType": str(r.exportType),
        "wikidata":   str(r.wd),
    })

df_all = pd.DataFrame(all_rows).dropna(subset=["lat","lon"])
print(f"Total items with Wikidata links: {len(df_all)}")
print(df_all["exportType"].value_counts().to_string())

In [ ]:
# ── World Map: All datasets combined ─────────────────────────────────────────
m_all = folium.Map(location=[30, 0], zoom_start=2, tiles="OpenStreetMap")

COLORS_MAP = {
    "ogham":      "red",
    "holywells":  "blue",
    "ci":         "purple",
    "benchmarks": "green",
    "sisal":      "orange",
    "romansites": "gray",
}

# One FeatureGroup per dataset for layer control
groups = {name: folium.FeatureGroup(name=name).add_to(m_all)
          for name in COLORS_MAP}

for _, row in df_all.iterrows():
    et = row["exportType"]
    color = COLORS_MAP.get(et, "cadetblue")
    wd_url = row["wikidata"].replace("http://wikidata.org/entity/",
                                     "https://www.wikidata.org/wiki/")
    popup_html = (f"<b>{row['label']}</b><br>"
                  f"Type: {et}<br>"
                  f'<a href="{wd_url}" target="_blank">Wikidata ↗</a>')

    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=6, color=color, fill=True, fill_opacity=0.8,
        popup=folium.Popup(popup_html, max_width=240),
        tooltip=f"{et}: {row['label']}"
    ).add_to(groups.get(et, m_all))

folium.LayerControl(collapsed=False).add_to(m_all)

# Legend
legend_items = "".join(
    f'<span style="color:{c};">●</span> {n}<br>'
    for n, c in COLORS_MAP.items()
)
legend_html = f"""
<div style="position:fixed;bottom:50px;left:30px;z-index:1000;
     background:white;padding:10px 15px;border-radius:6px;
     border:1px solid #ccc;font-size:13px;line-height:1.8;">
  <b>osm2lod datasets</b><br>(Wikidata-linked only)<br>
  {legend_items}
</div>"""
m_all.get_root().html.add_child(folium.Element(legend_html))

display(HTML(f"<p><b>All datasets – Wikidata-linked items</b>: {len(df_all)} entities from "
             f"{df_all['exportType'].nunique()} datasets</p>"))
m_all

---
## 9 — SPARQL Queries for QGIS / Unicorn Plugin

The following queries can be pasted directly into the **SPARQL Unicorn QGIS Plugin** (or any SPARQL endpoint loaded with these TTL files via an in-memory triplestore like Apache Jena Fuseki).

> **Tip:** Load all TTL files into a single named graph in Fuseki, then run these queries against `http://localhost:3030/osm2lod/sparql`.

In [ ]:
queries = {

"1_all_items_with_coords": """
# All osm2lod items with coordinates (for QGIS point layer)
PREFIX osm2lod:  <http://research-squirrel-engineers.github.io/osm2lod/>
PREFIX geosparql:<http://www.opengis.net/ont/geosparql#>
PREFIX rdfs:     <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?item ?label ?exportType ?wkt
WHERE {
  ?item osm2lod:exportType ?exportType ;
        geosparql:hasGeometry ?geom .
  ?geom geosparql:asWKT ?wkt .
  OPTIONAL { ?item rdfs:label ?label . FILTER(lang(?label) = "en") }
}
ORDER BY ?exportType ?label
""",

"2_wikidata_linked": """
# Items with Wikidata sameAs link — federation target for enrichment
PREFIX osm2lod:  <http://research-squirrel-engineers.github.io/osm2lod/>
PREFIX geosparql:<http://www.opengis.net/ont/geosparql#>
PREFIX rdfs:     <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl:      <http://www.w3.org/2002/07/owl#>

SELECT ?item ?label ?exportType ?wkt ?wikidata
WHERE {
  ?item osm2lod:exportType ?exportType ;
        geosparql:hasGeometry ?geom ;
        owl:sameAs ?wikidata .
  ?geom geosparql:asWKT ?wkt .
  FILTER(STRSTARTS(STR(?wikidata), "http://wikidata.org/"))
  OPTIONAL { ?item rdfs:label ?label . FILTER(lang(?label) = "en") }
}
""",

"3_holywells_denomination": """
# Holy Wells grouped by denomination
PREFIX osm2lod:  <http://research-squirrel-engineers.github.io/osm2lod/>
PREFIX geosparql:<http://www.opengis.net/ont/geosparql#>
PREFIX rdfs:     <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?label ?wkt ?denomination
WHERE {
  ?item a osm2lod:HolyWell ;
        geosparql:hasGeometry ?geom .
  ?geom geosparql:asWKT ?wkt .
  OPTIONAL { ?item rdfs:label ?label . FILTER(lang(?label) = "en") }
  OPTIONAL { ?item osm2lod:osmtag__denomination ?denomination }
}
""",

"4_sisal_global": """
# SISAL cave sites — global distribution
PREFIX osm2lod:  <http://research-squirrel-engineers.github.io/osm2lod/>
PREFIX geosparql:<http://www.opengis.net/ont/geosparql#>
PREFIX rdfs:     <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl:      <http://www.w3.org/2002/07/owl#>

SELECT ?label ?wkt ?wikidata ?website
WHERE {
  ?item a osm2lod:SisalSite ;
        geosparql:hasGeometry ?geom .
  ?geom geosparql:asWKT ?wkt .
  OPTIONAL { ?item rdfs:label ?label . FILTER(lang(?label) = "en") }
  OPTIONAL { ?item owl:sameAs ?wikidata .
             FILTER(STRSTARTS(STR(?wikidata), "http://wikidata.org/")) }
  OPTIONAL { ?item osm2lod:osmtag__website ?website }
}
""",

"5_ogham_wikidata": """
# Ogham stones — with and without Wikidata (completeness check)
PREFIX osm2lod:  <http://research-squirrel-engineers.github.io/osm2lod/>
PREFIX geosparql:<http://www.opengis.net/ont/geosparql#>
PREFIX rdfs:     <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl:      <http://www.w3.org/2002/07/owl#>

SELECT ?label ?wkt (BOUND(?wd) AS ?has_wikidata) ?wd
WHERE {
  ?item a osm2lod:OghamStone ;
        geosparql:hasGeometry ?geom .
  ?geom geosparql:asWKT ?wkt .
  OPTIONAL { ?item rdfs:label ?label . FILTER(lang(?label) = "en") }
  OPTIONAL { ?item owl:sameAs ?wd .
             FILTER(STRSTARTS(STR(?wd), "http://wikidata.org/")) }
}
""",

}

print("=" * 60)
print("SPARQL QUERIES FOR QGIS UNICORN / FUSEKI")
print("=" * 60)
for name, q in queries.items():
    print(f"\n{'─'*60}")
    print(f"Query: {name}")
    print(f"{'─'*60}")
    print(q.strip())

---
## 10 — Summary: The Linked Data Argument

| Dataset | Items | Wikidata links | Coverage |
|---------|------:|---------------:|---------:|
| Holy Wells | 734 | 138 | 19% |
| SISAL Sites | 221 | 64 | 29% |
| Ogham Stones | 107 | 58 | 54% |
| Benchmarks | 21 | 0 | 0% |
| Roman Sites | 15 | 9 | 60% |
| CI Findspots | 11 | 10 | 91% |

**Key points:**
- All six datasets share the same RDF vocabulary (`osm2lod:`, `geosparql:`, `owl:sameAs`) — enabling cross-dataset queries **without any additional ETL**.
- `owl:sameAs` links to Wikidata open up federation with one of the world's largest open knowledge graphs.
- GeoSPARQL `asWKT` literals make the data directly consumable by spatial tools (QGIS Unicorn, GeoTriples, Apache Jena GeoSPARQL).
- The Wikibase import (via QuickStatements) creates a queryable, versioned, resolvable knowledge graph from otherwise static OSM snapshots.